# Phase 7 (Prior Case Retrieval) — Run #4: Phase-1 **Decoder-only model 03** (ALiBi + Multi-Head Latent Attention, 13 blocks) as a bi-encoder, with **partial / layer-wise fine-tuning**

Same pipeline as runs #1–#3 (data, views, token cache, BM25 lists, objective, fusion, selection, analysis, exports); the backbone and the technique change. Edit **Cell 2** only.

### Partial fine-tuning, and why it suits this task
Only the **top `top_k` = 4 of the 13 blocks** (plus the final norm and the identity-initialised head) are trained; the embedding and the lower 9 blocks stay exactly as in Phase 1. Learning rates **decay layer by layer downwards**, `lr_i = lr · decay^{(n−1−i)}` (decay 0.8: the top block gets `lr`, the 4th from the top `0.51·lr`), following the usual picture of a pretrained stack (Yosinski et al. 2014; ULMFiT's discriminative rates): lower blocks hold general lexical/syntactic features worth keeping, upper blocks hold the task-specific composition that retrieval has to re-shape.
Two practical consequences:
* **Regularisation.** Runs #1–#3 are limited by generalisation from 827 queries, not by optimisation. Freezing 9 blocks removes ~70 % of the free parameters while keeping full-rank updates where they matter.
* **Speed.** The frozen lower blocks run under `torch.no_grad()`: no activations are stored for them and the backward pass stops at block 9 — about twice as fast per step as a full backward.

### Bidirectional top blocks (`ATTN_MODE = "bidir_top"`)
The trained top blocks can drop the causal mask: block `i ≥ 9` then uses a **symmetric** ALiBi bias `−m_h·|i − j|` (right context visible, the same distance penalty in both directions) with a key-padding mask, while the frozen lower blocks keep their Phase-1 causal bias. The lower blocks still compute left-to-right features, but the top 4 can mix them across the whole chunk, so **every** position of the final layer summarises the full chunk and **mean pooling** becomes the natural readout (the causal arguments for `wmean`/`last` disappear). This is the same hybrid as in the Phase-5 Dec-03 run. It changes the function at step 0, so the notebook measures three points: zero-shot (causal, the Phase-1 function, all poolings), **step 0 in `bidir_top`** (how much does unmasking the top blocks disturb the representation before any training?), and after training. `ATTN_MODE = "causal"` keeps the Phase-1 mask everywhere (then pooling is chosen zero-shot as in runs #1/#3).

### ALiBi and MLA, briefly
* **ALiBi** adds `−m_h (i − j)` to the attention logits of head `h` instead of position embeddings, i.e. it multiplies attention by `exp(−m_h·distance)`: each head has a fixed "attention span" set by its slope. No position table means chunks could be longer than the 1024 training length (not used here, for comparability).
* **MLA** compresses keys and values of all heads into one shared latent `c_t = W_DKV x_t` and expands it per head (`k = W_UK c_t`, `v = W_UV c_t`), a low-rank factorisation of the K/V projections that was designed for a small KV cache. For a bi-encoder there is no cache; MLA is simply this model's attention.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle, copy
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import contextlib, hashlib

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p7_dec03_partial"
METHOD     = "partial"        # Phase-7 list: Partial / layer-wise fine-tuning
SMOKE_TEST = False            # True: SMOKE_STEPS steps, dev on SMOKE_DEV_Q queries (full dev pool), NO test / export
ATTN_MODE  = "bidir_top"      # "bidir_top" (trained top blocks bidirectional, symmetric ALiBi) | "causal" (Phase-1 mask everywhere)

# ---- Phase-1 checkpoint (caps_decoder_only_architecture_03.ipynb: runs/<ARCH>/best.pt), same file as Phases 3-6 -----------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-dec-03-best/best.pt") if IS_KAGGLE else Path("./input/best_dec03.pt")

# ---- Data + audit artifacts (same as runs #1-#3) ---------------------------------------------------
HF_REPO        = "Exploration-Lab/CS-781-Capstone"
LOCAL_DATA_DIR = None
P7    = "Phase_7_PCR"
FILES = {"train": ("Train/train.json", "Train/train_candidates.json"),
         "dev":   ("Dev/dev.json",     "Dev/dev_candidates.json"),
         "test":  ("Test/test.json",   "Test/test_candidates.json")}
BM25_PATH   = Path("/kaggle/input/datasets/ascyrax/caps-p7-bm25/bm25_top300.json") if IS_KAGGLE else Path("./input/bm25_top300.json")
TOKCACHE_IN = Path("/kaggle/input/datasets/ascyrax/caps-p7-tokcache") if IS_KAGGLE else Path("./input/tokcache")
N_PROC      = 4

# ---- Views (identical to runs #1-#3) ---------------------------------------------------------------
CHUNK_TOKENS   = 512          # [BOS] + 510 text tokens + [EOS]
MAX_DOC_CHUNKS = 16
QUERY_VIEW     = "ctx+head"
CTX_WINDOW     = 1
Q_MAX_CHUNKS   = 4
CIT_TAG        = "<CITATION>"

# ---- Encoder / scoring ----------------------------------------------------------------------------
POOLING    = "auto"           # "mean" | "wmean" | "last" | "auto". With ATTN_MODE="bidir_top", "auto" -> "mean" for training
SCORE_MODE = "auto"           # "mean" | "maxsim" | "auto"
USE_HEAD   = True
TAU        = 0.05
EMB_BATCH  = 64

# ---- Training -------------------------------------------------------------------------------------
EPOCHS        = 4
BATCH_PAIRS   = 32
C_TRAIN       = 4
QC_TRAIN      = 2
HN_TOP        = 50
FWD_MICRO     = 32
WARMUP_FRAC   = 0.06
GRAD_CLIP     = 1.0
GRAD_CKPT     = True          # applies to the trained top blocks (the frozen ones run under no_grad and store nothing)
TIME_BUDGET_MIN = 360
HP = {"partial": dict(lr=1e-4, head_lr=2.5e-4, wd=0.01, top_k=4, decay=0.8)}   # top_k of the 13 blocks; lr x decay per block downwards
POS_SAMPLING  = "uniform"     # "uniform" | "bm25_hard" (see run #3)
HARD_RANK, HARD_BOOST = 50, 2.0

# ---- Fusion / evaluation --------------------------------------------------------------------------
K_EVAL     = 7
ALPHA_GRID = tuple(np.round(np.arange(0.0, 1.0001, 0.05), 2))
RRF_K      = 60
RECALL_KS  = (7, 20, 50, 100, 300)

# ---- Exports --------------------------------------------------------------------------------------
EXPORT_TEACHER = True
SAVE_SCORES    = True
SAVE_TOP       = 300

# ---- Smoke test -----------------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_DEV_Q = 40, 40

# ---- derived --------------------------------------------------------------------------------------
assert METHOD in HP and POOLING in ("mean", "wmean", "last", "auto") and SCORE_MODE in ("mean", "maxsim", "auto")
assert QUERY_VIEW in ("ctx", "head", "ctx+head") and POS_SAMPLING in ("uniform", "bm25_hard") and ATTN_MODE in ("bidir_top", "causal")
PARTIAL_K = HP["partial"]["top_k"]                # (used by the checkpoint-loading cell)
BIDIR_TOP_K = PARTIAL_K if ATTN_MODE == "bidir_top" else 0
BODY = CHUNK_TOKENS - 2
MAX_DOC_TOK = MAX_DOC_CHUNKS * BODY
MAX_DOC_CHARS = int(MAX_DOC_TOK * 4.6)
RUN_DIR = WORK_DIR / "runs" / RUN_NAME; RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | {METHOD} {HP[METHOD]} | attention {ATTN_MODE} (top {BIDIR_TOP_K}) | chunks {CHUNK_TOKENS} x <= {MAX_DOC_CHUNKS} | "
      f"B={BATCH_PAIRS} pairs, C={C_TRAIN}, QC={QC_TRAIN}, tau={TAU} | epochs {EPOCHS} | pos sampling {POS_SAMPLING} | smoke={SMOKE_TEST}")

RUN p7_dec03_partial | partial {'lr': 0.0001, 'head_lr': 0.00025, 'wd': 0.01, 'top_k': 4, 'decay': 0.8} | attention bidir_top (top 4) | chunks 512 x <= 16 | B=32 pairs, C=4, QC=2, tau=0.05 | epochs 4 | pos sampling uniform | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- as in Phases 3-4.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_decoder_only_architecture_03.ipynb` (Cells 4-11)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# Cell 4 — Model configuration
#
# One config drives three architectures (all registered, all testable):
#   preset_alibi_mla()  -> the NEW architecture (ALiBi positions + Multi-Head Latent Attention)
#   preset_alibi_mha()  -> ablation: ALiBi with ordinary MHA (isolates the ALiBi effect)
#   preset_rope_mha()   -> corrected baseline: RoPE + MHA (model 01 accidentally had NO positions)

@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 13
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                # only used by mixer_type="attention" (GQA if < n_head)
    head_dim: int = 64
    ff_mult: float = 8/3              # SwiGLU hidden = round64(ff_mult * d_model) = 1408
    d_ff: Optional[int] = None
    max_seq_len: int = 1024           # TRAINING context. ALiBi itself has no length limit.

    # --- components ---
    norm_type: str = "rmsnorm"        # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "alibi"           # "alibi" | "rope" | "learned" | "none"
    rope_theta: float = 10000.0
    mixer_type: str = "mla"           # "attention" (MHA/GQA) | "mla"
    kv_latent_dim: int = 128          # d_c: MLA caches only this many numbers per token per layer
    mla_latent_norm: bool = True      # RMSNorm on the latent c_t (as in DeepSeek-V2)
    mlp_type: str = "swiglu"          # "swiglu" | "gelu"
    channel_mixer_type: str = "mlp"

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("alibi", "rope", "learned", "none")
        assert self.mixer_type in ("attention", "mla")
        if self.mixer_type == "mla":
            # RoPE rotates q and k by a position-dependent matrix that sits BETWEEN W_Q and
            # W_UK, which breaks MLA's weight absorption (DeepSeek needs "decoupled RoPE").
            # ALiBi only adds a bias outside q.k, so it composes cleanly with MLA.
            assert self.pos_type != "rope", "MLA + RoPE needs decoupled RoPE; use ALiBi"
            assert not self.bias, "MLA absorption assumes bias-free up-projections"
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


def preset_alibi_mla(vocab_size=None, max_seq_len=1024):
    # MLA saves ~0.33M attention params per layer vs MHA (d_c=128). 13 layers instead of 12
    # reinvests that saving so the model stays parameter-matched to the 12-layer baseline.
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=13, pos_type="alibi",
                       mixer_type="mla", kv_latent_dim=128, max_seq_len=max_seq_len)

def preset_alibi_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="alibi",
                       mixer_type="attention", max_seq_len=max_seq_len)

def preset_rope_mha(vocab_size=None, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, n_layer=12, pos_type="rope",
                       mixer_type="attention", max_seq_len=max_seq_len)

PRESETS = {"alibi_mla": preset_alibi_mla, "alibi_mha": preset_alibi_mha, "rope_mha": preset_rope_mha}
print(json.dumps(asdict(preset_alibi_mla()), indent=2))

{
  "vocab_size": 32000,
  "n_layer": 13,
  "d_model": 512,
  "n_head": 8,
  "n_kv_head": 8,
  "head_dim": 64,
  "ff_mult": 2.6666666666666665,
  "d_ff": 1408,
  "max_seq_len": 1024,
  "norm_type": "rmsnorm",
  "norm_eps": 1e-05,
  "pos_type": "alibi",
  "rope_theta": 10000.0,
  "mixer_type": "mla",
  "kv_latent_dim": 128,
  "mla_latent_norm": true,
  "mlp_type": "swiglu",
  "channel_mixer_type": "mlp",
  "bias": false,
  "attn_dropout": 0.0,
  "resid_dropout": 0.0,
  "embd_dropout": 0.0,
  "tie_embeddings": true,
  "init_std": 0.02,
  "use_sdpa": true
}


In [6]:
# Cell 5 — Normalisation layers (hand-written)

class LayerNorm(nn.Module):
    """Classic LayerNorm: centre by the mean, scale by the std, learn gain g and bias b."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return (((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype)) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: no mean-centring and no bias; x / rms(x) * g. Statistics computed in fp32."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g


def build_norm(cfg: ModelConfig, d: Optional[int] = None):
    d = d or cfg.d_model
    if cfg.norm_type == "layernorm":
        return LayerNorm(d, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(d, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [7]:
# Cell 6 — Positional mechanisms: ALiBi (new) and RoPE (fixed wiring)
#
# ALiBi (Press et al., 2022). No position embeddings at all. Each head h adds a fixed bias
#       s_ij = q_i.k_j / sqrt(d)  -  m_h * (i - j),        j <= i
# Inside the softmax this multiplies attention by exp(-m_h (i-j)): an exponential recency prior.
# Only the distance (i - j) matters: softmax ignores a per-row constant, and
# -m(i-j) = -m*i + m*j, where -m*i is constant along row i.
# Slopes are geometric, m_h = 2^(-8h/H), h = 1..H. For H = 8: 1/2, 1/4, ..., 1/256, so heads
# range from "looks only a few tokens back" to "looks hundreds of tokens back".

def alibi_slopes(n_head: int) -> List[float]:
    def pow2_slopes(n):
        start = 2.0 ** (-8.0 / n)
        return [start ** (i + 1) for i in range(n)]
    if math.log2(n_head).is_integer():
        return pow2_slopes(n_head)
    # Non-power-of-2 head counts: the paper's interpolation rule.
    c = 2 ** math.floor(math.log2(n_head))
    return pow2_slopes(c) + pow2_slopes(2 * c)[0::2][: n_head - c]


class ALiBi(nn.Module):
    def __init__(self, n_head: int):
        super().__init__()
        self.register_buffer("slopes", torch.tensor(alibi_slopes(n_head), dtype=torch.float32),
                             persistent=False)

    def bias(self, q_len: int, k_len: int, device) -> torch.Tensor:
        """[1, H, q_len, k_len] float32. Queries are the LAST q_len of the k_len positions
        (so it also works for KV-cached decoding). Includes the causal mask (-inf for j > i)."""
        q_pos = torch.arange(k_len - q_len, k_len, device=device)
        k_pos = torch.arange(k_len, device=device)
        dist = (q_pos[:, None] - k_pos[None, :]).float()          # i - j
        b = -self.slopes.to(device)[:, None, None] * dist          # [H, Tq, Tk]
        b = b.masked_fill(dist < 0, float("-inf"))
        return b.unsqueeze(0)


class RoPE(nn.Module):
    """Rotary embeddings. Table is built for 2x the training length so generation never overflows."""
    def __init__(self, head_dim: int, max_pos: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_pos).float(), inv_freq)   # [P, D/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def tables(self, position_ids: torch.Tensor):
        """position_ids [B, T] -> (cos, sin) each [B, 1, T, D/2]."""
        return self.cos[position_ids].unsqueeze(1), self.sin[position_ids].unsqueeze(1)

    @staticmethod
    def apply(x, cos, sin):
        """x [B, H, T, D]; rotation done in fp32 for stability."""
        dt = x.dtype
        x = x.float()
        x1, x2 = x[..., 0::2], x[..., 1::2]
        out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)
        return out.to(dt)

In [8]:
# Cell 7 — Shared attention kernel + standard MHA/GQA token mixer
#
# The Backbone builds ONE additive mask per forward pass (`attn_bias`, shape [B or 1, H or 1, T, S])
# containing ALiBi + causal + padding, and passes it to every layer. Every token mixer then
# calls `attend`. When attn_bias is None we use SDPA's fast is_causal path.

def attend(q, k, v, attn_bias, is_causal, scale, dropout_p=0.0, use_sdpa=True):
    """q [B,H,T,Dq], k [B,H,S,Dq], v [B,H,S,Dv] -> [B,H,T,Dv]."""
    if attn_bias is not None:
        attn_bias = attn_bias.to(q.dtype)          # SDPA requires mask dtype == query dtype
        is_causal = False
    if use_sdpa:
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                                              dropout_p=dropout_p, is_causal=is_causal,
                                              scale=scale)
    # Manual reference path (used by the tests to cross-check SDPA).
    scores = (q.float() @ k.float().transpose(-2, -1)) * scale
    if is_causal:
        T, S = q.shape[-2], k.shape[-2]
        scores = scores.masked_fill(
            torch.triu(torch.ones(T, S, dtype=torch.bool, device=q.device), diagonal=S - T + 1),
            float("-inf"))
    if attn_bias is not None:
        scores = scores + attn_bias.float()
    p = F.softmax(scores, dim=-1)
    if dropout_p > 0:
        p = F.dropout(p, dropout_p)
    return (p @ v.float()).to(q.dtype)


class Attention(nn.Module):
    """Multi-head (n_kv_head == n_head) or grouped-query (n_kv_head < n_head) self-attention.
    Cache format: (k, v), each [B, n_kv_head, S, head_dim]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)

    @staticmethod
    def cache_len(cache):
        return cache[0].shape[2]

    def cache_numbers_per_token(self):
        return 2 * self.n_kv_head * self.head_dim

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)
        if rope_cs is not None:                       # FIX: RoPE is now actually applied
            cos, sin = rope_cs
            q, k = RoPE.apply(q, cos, sin), RoPE.apply(k, cos, sin)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k, v = k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)
        out = attend(q, k, v, attn_bias, is_causal=(attn_bias is None and T > 1),
                     scale=self.scale,
                     dropout_p=self.cfg.attn_dropout if self.training else 0.0,
                     use_sdpa=self.cfg.use_sdpa)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [9]:
# Cell 8 — Multi-Head Latent Attention (MLA, DeepSeek-V2), adapted to ALiBi
#
#   c_t      = Norm(W_DKV x_t)          in R^{d_c}        (the ONLY thing we cache)
#   k_t^(h)  = W_UK^(h) c_t,   v_t^(h) = W_UV^(h) c_t    (every head gets its own K, V)
#   q_t^(h)  = W_Q^(h) x_t
#
# KV cache per token per layer: MHA stores 2*H*D_h = 1024 numbers; MLA stores d_c = 128 (8x less).
#
# MLA vs GQA: GQA *ties* heads (groups share identical K,V). MLA gives each head distinct K,V,
# all reconstructed from a shared rank-d_c subspace. With d_c = 2*n_kv*D_h, MLA can exactly
# express GQA (the up-projections just copy slices), so at equal cache size GQA is a special case.
#
# Weight absorption (inference trick, implemented below as `absorb=True`):
#   q_i^(h).k_j^(h) = (W_Q^(h) x_i)^T W_UK^(h) c_j = [W_UK^(h)T q_i^(h)] . c_j
# so we can attend directly against the cached latents without ever building K:
#   q_lat = W_UK^T q,  scores = q_lat . c,  out_lat = softmax(scores) c,  out^(h) = W_UV^(h) out_lat
# RoPE would insert a position-dependent rotation R_{i-j} between W_Q and W_UK and break this;
# ALiBi's bias lives outside the dot product, so absorption is unaffected.
# Training uses the materialised path (simpler, identical maths); a test checks both agree.

class MLAttention(nn.Module):
    """Cache format: c, [B, S, d_c]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        H, Dh, dc = cfg.n_head, cfg.head_dim, cfg.kv_latent_dim
        self.n_head, self.head_dim, self.d_c = H, Dh, dc
        self.scale = 1.0 / math.sqrt(Dh)              # scores are still per-head q.k with dim D_h
        self.q_proj = nn.Linear(cfg.d_model, H * Dh, bias=False)
        self.kv_down = nn.Linear(cfg.d_model, dc, bias=False)          # W_DKV
        self.kv_norm = build_norm(cfg, dc) if cfg.mla_latent_norm else nn.Identity()
        self.k_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UK (all heads)
        self.v_up = nn.Linear(dc, H * Dh, bias=False)                  # W_UV (all heads)
        self.o_proj = nn.Linear(H * Dh, cfg.d_model, bias=False)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)
        self.absorb = False                           # runtime switch (inference only)

    @staticmethod
    def cache_len(cache):
        return cache.shape[1]

    def cache_numbers_per_token(self):
        return self.d_c

    def forward(self, x, attn_bias=None, rope_cs=None, kv_cache=None):
        assert rope_cs is None, "MLA is used with ALiBi (or no positions), not RoPE"
        B, T, _ = x.shape
        H, Dh = self.n_head, self.head_dim
        c = self.kv_norm(self.kv_down(x))                         # [B, T, d_c]
        if kv_cache is not None:
            c = torch.cat([kv_cache, c], dim=1)                   # [B, S, d_c]
        new_cache = c
        S = c.shape[1]
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)      # [B, H, T, Dh]
        is_causal = (attn_bias is None and T > 1)
        dp = self.cfg.attn_dropout if self.training else 0.0

        if not self.absorb:
            k = self.k_up(c).view(B, S, H, Dh).transpose(1, 2)    # [B, H, S, Dh]
            v = self.v_up(c).view(B, S, H, Dh).transpose(1, 2)
            out = attend(q, k, v, attn_bias, is_causal, self.scale, dp, self.cfg.use_sdpa)
        else:
            W_uk = self.k_up.weight.view(H, Dh, self.d_c)         # rows of head h -> W_UK^(h)
            W_uv = self.v_up.weight.view(H, Dh, self.d_c)
            q_lat = torch.einsum("bhtd,hdc->bhtc", q, W_uk.to(q.dtype))   # W_UK^T q
            c_h = c.unsqueeze(1).expand(B, H, S, self.d_c)
            out_lat = attend(q_lat, c_h, c_h, attn_bias, is_causal, self.scale, dp,
                             self.cfg.use_sdpa)                   # [B, H, T, d_c]
            out = torch.einsum("bhtc,hdc->bhtd", out_lat, W_uv.to(out_lat.dtype))

        out = out.transpose(1, 2).contiguous().view(B, T, H * Dh)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [10]:
# Cell 9 — Channel mixers, registries, transformer block

class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))


TOKEN_MIXERS: Dict[str, Any] = {}
CHANNEL_MIXERS: Dict[str, Any] = {}

def register_token_mixer(name):
    def deco(fn):
        TOKEN_MIXERS[name] = fn
        return fn
    return deco

def register_channel_mixer(name):
    def deco(fn):
        CHANNEL_MIXERS[name] = fn
        return fn
    return deco

# FIX: token mixers no longer receive `rope` at build time (model 01 silently dropped it there).
# Positional information now flows through forward(): rope_cs / attn_bias, built in the Backbone.
@register_token_mixer("attention")
def _build_attention(cfg):
    return Attention(cfg)

@register_token_mixer("mla")
def _build_mla(cfg):
    return MLAttention(cfg)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    return {"swiglu": SwiGLU, "gelu": GELUMLP}[cfg.mlp_type](cfg)


class Block(nn.Module):
    """Pre-norm block: x = x + mixer(norm(x)); x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg)
        self.norm2 = build_norm(cfg)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)

    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [11]:
# Cell 10 — Backbone and language model

class Backbone(nn.Module):
    """Embedding -> N blocks -> final norm. Builds positional info and masks ONCE per forward."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.alibi = ALiBi(cfg.n_head) if cfg.pos_type == "alibi" else None
        self.rope = RoPE(cfg.head_dim, 2 * cfg.max_seq_len, cfg.rope_theta) \
            if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(2 * cfg.max_seq_len, cfg.d_model) \
            if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def _attn_bias(self, B, T, past, attention_mask, device):
        """Additive mask [B|1, H|1, T, S] or None (=> SDPA fast causal path)."""
        S = past + T
        need = (self.alibi is not None) or (attention_mask is not None) or (past > 0 and T > 1)
        if not need:
            return None
        if self.alibi is not None:
            bias = self.alibi.bias(T, S, device)                          # ALiBi + causal
        else:
            q_pos = torch.arange(past, S, device=device)
            k_pos = torch.arange(S, device=device)
            bias = torch.zeros(T, S, device=device).masked_fill(
                k_pos[None, :] > q_pos[:, None], float("-inf"))[None, None]
        if attention_mask is not None:
            # Mask padded KEYS. A padded QUERY would then have every key masked -> softmax of
            # all -inf -> NaN, which can leak into later layers through 0*NaN. So let each
            # padded query attend to itself (its output is never used).
            key_pad = ~attention_mask.bool()                                   # [B, S]
            q_idx = torch.arange(past, S, device=device)
            k_idx = torch.arange(S, device=device)
            is_self = q_idx[:, None] == k_idx[None, :]                         # [T, S]
            block = key_pad[:, None, :] & ~is_self[None]                       # [B, T, S]
            pad = torch.zeros(B, 1, T, S, device=device).masked_fill(block[:, None], float("-inf"))
            bias = bias + pad
        return bias

    def forward(self, input_ids, attention_mask=None, kv_caches=None):
        B, T = input_ids.shape
        dev = input_ids.device
        past = 0 if kv_caches is None else self.blocks[0].mixer.cache_len(kv_caches[0])
        x = self.tok_emb(input_ids)

        rope_cs = None
        if self.rope is not None or self.pos_emb is not None:
            if attention_mask is not None:       # left padding: real tokens start at position 0
                pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, -T:]
            else:
                pos = torch.arange(past, past + T, device=dev).unsqueeze(0).expand(B, T)
            if self.rope is not None:
                rope_cs = self.rope.tables(pos)
            else:
                x = x + self.pos_emb(pos)

        attn_bias = self._attn_bias(B, T, past, attention_mask, dev)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, attn_bias=attn_bias, rope_cs=rope_cs,
                        kv_cache=None if kv_caches is None else kv_caches[i])
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def set_mla_absorb(self, flag: bool):
        for blk in self.backbone.blocks:
            if isinstance(blk.mixer, MLAttention):
                blk.mixer.absorb = flag

    def forward(self, input_ids, targets=None, attention_mask=None, kv_caches=None,
                last_only=False, reduction="mean"):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      kv_caches=kv_caches)
        if last_only:                 # generation: only the last position's logits are needed
            h = h[:, -1:, :]
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=-100, reduction=reduction)
            if reduction == "none":
                loss = loss.view(targets.shape)
        return logits, loss, new_caches

In [12]:
# Cell 11 — Init, parameter counting, optimiser groups, checkpoint I/O

def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, 0.02) everywhere; residual-branch output projections scaled by 1/sqrt(2L)
    so the residual stream's variance does not grow with depth."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            blk.mixer.o_proj.weight.mul_(scale)
            blk.cmix.down_proj.weight.mul_(scale)


def count_params(model: LegalLM) -> Dict[str, int]:
    seen, total = set(), 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p))
            total += p.numel()
    emb = model.backbone.tok_emb.weight.numel()
    attn = sum(p.numel() for blk in model.backbone.blocks for p in blk.mixer.parameters())
    return {"total": total, "embedding": emb, "non_embedding": total - emb, "attention": attn}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Decay only 2-D weight matrices (not embeddings, norm gains or biases)."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name and "pos_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model: LegalLM, cfg: ModelConfig, extra: dict = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, path)


def load_model(path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

## Load the Phase-1 checkpoint

In [13]:
# Cell 12 — Load the Phase-1 checkpoint and sanity-check it
_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
lm_cfg = ModelConfig(**_ck["config"])
lm = LegalLM(lm_cfg)
lm.load_state_dict(_ck["model"])
lm_extra = _ck.get("extra", {})
del _ck
lm = lm.to(device).eval()
pc = count_params(lm)
print(f"Phase-1 model | pos_type={lm_cfg.pos_type} mixer={lm_cfg.mixer_type} (latent {lm_cfg.kv_latent_dim}) | "
      f"layers={lm_cfg.n_layer} d_model={lm_cfg.d_model} heads={lm_cfg.n_head} d_ff={lm_cfg.d_ff} | "
      f"params {pc['total']/1e6:.2f}M (embedding {pc['embedding']/1e6:.2f}M)")
print("checkpoint extra: step", lm_extra.get("step"), "| val_loss", lm_extra.get("val_loss"))

# If the weights did not load properly, the LM loss on a normal legal sentence would sit near ln(V) ~ 10.4.
_sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
_ids = [BOS_ID] + tokenizer.encode(_sent)
_x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l, _ = lm(_x, targets=_y)
print(f"LM loss on a legal sentence: {_l.item():.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
assert _l.item() < 9.0, "checkpoint looks untrained / mis-loaded"

if ATTN_MODE in ("bidir", "bidir_top") and lm_cfg.pos_type != "alibi":
    raise RuntimeError(f"ATTN_MODE='{ATTN_MODE}' is implemented for ALiBi checkpoints (pos_type='alibi'); this one has "
                       f"pos_type='{lm_cfg.pos_type}'. Use ATTN_MODE='causal'.")
if METHOD == "partial":
    assert 0 <= PARTIAL_K <= lm_cfg.n_layer, "PARTIAL_K must be between 0 and the number of layers"
if lm_cfg.pos_type != "alibi":
    raise RuntimeError(f"this notebook handles the ALiBi checkpoints of Decoder-03 (alibi_mla / alibi_mha); got pos_type='{lm_cfg.pos_type}'. "
                       f"For a RoPE or NoPE checkpoint use the Dec-02 / Dec-01 notebooks.")

Phase-1 model | pos_type=alibi mixer=mla (latent 128) | layers=13 d_model=512 heads=8 d_ff=1408 | params 53.89M (embedding 16.38M)
checkpoint extra: step 4591 | val_loss 1.8845605307433289
LM loss on a legal sentence: 1.947  (ln V = 10.37)


## Phase-7 data (the audit's loader) and the BM25 lists
The loader is the audit's (Cell 4), unchanged. IDs are matched on a canonical key (leading zeros stripped); every output uses the pool's own ID string. The BM25 lists come from the audit's Cell 12 (`bm25_top300.json`, best configuration, no year filter).

In [14]:
# Cell 13 — Load the 6 files (audit loader) + BM25 lists
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

def fetch(rel):
    rel = f"{P7}/{rel}"
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet, or download {P7} and set LOCAL_DATA_DIR.") from ex

def read_json_any(path):
    txt = Path(path).read_text(encoding="utf-8")
    try: return json.loads(txt)
    except json.JSONDecodeError: return [json.loads(l) for l in txt.splitlines() if l.strip()]

ID_KEYS   = ("id", "query_id", "case_id", "candidate_id", "doc_id", "document_id", "qid", "cid")
TEXT_KEYS = ("text", "case_text", "document", "content", "sentences", "judgment", "judgement", "body")
GOLD_KEYS = ("relevant_candidates", "relevant", "relevant_ids", "positives", "labels", "gold", "citations", "cited")
def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default
def as_sents(v):
    if isinstance(v, str): return [s for s in v.split("\n")]
    out = []
    for x in v:
        if isinstance(x, list): out.extend(str(y) for y in x)
        elif isinstance(x, dict): out.append(str(_first(x, TEXT_KEYS + ("sentence",), "")))
        else: out.append(str(x))
    return out
def canon(x): s = str(x).strip(); return s.lstrip("0") or "0"
def unwrap(raw):
    if isinstance(raw, dict) and len(raw) == 1:
        v = next(iter(raw.values()))
        if isinstance(v, (list, dict)) and len(v) > 1: return v
    if isinstance(raw, dict) and isinstance(raw.get("data"), list): return raw["data"]
    return raw
def items_of(raw):
    raw = unwrap(raw)
    if isinstance(raw, dict):
        out = []
        for k, v in raw.items():
            if isinstance(v, dict): v = dict(v); v.setdefault("id", k)
            else: v = {"id": k, "text": v}
            out.append(v)
        return out
    return list(raw)

DOCS = {}                      # canonical id -> {"id": original string, "sents": [...]}, shared by all splits (pools overlap)
POOL, QRY = {}, {}             # split -> list of canonical ids (pool order) ; split -> list of query dicts
for split, (qf, cf) in FILES.items():
    raw_c = read_json_any(fetch(cf)); order, seen = [], set()
    for e in items_of(raw_c):
        k = canon(_first(e, ID_KEYS))
        if k in seen: continue
        seen.add(k); order.append(k)
        if k not in DOCS: DOCS[k] = dict(id=str(_first(e, ID_KEYS)), sents=as_sents(_first(e, TEXT_KEYS)))
    POOL[split] = order; del raw_c
    raw_q = read_json_any(fetch(qf)); qs = []
    pool_set = set(order)
    for i, e in enumerate(items_of(raw_q)):
        k = canon(_first(e, ID_KEYS, i))
        if k not in DOCS: DOCS[k] = dict(id=str(_first(e, ID_KEYS, i)), sents=as_sents(_first(e, TEXT_KEYS)))
        g = _first(e, GOLD_KEYS)
        gold = None
        if split != "test":
            g = [g] if isinstance(g, (str, int)) else (g or [])
            gold = []
            for x in g:
                c = canon(x)
                if c in pool_set and c not in gold: gold.append(c)
        qs.append(dict(key=k, id=str(_first(e, ID_KEYS, i)), gold=gold))
    QRY[split] = qs; del raw_q; gc.collect()
    print(f"{split:5s}: {len(qs)} queries | pool {len(order)} | queries inside their own pool: {sum(q['key'] in pool_set for q in qs)}"
          + (f" | gold pairs {sum(len(q['gold']) for q in qs)} | queries without gold {sum(not q['gold'] for q in qs)}" if split != "test" else ""))
print("distinct documents over all splits:", len(DOCS))

_bm = json.loads(BM25_PATH.read_text())
BM25_CFG, BM25 = _bm["config"], {s: {canon(q): [(canon(c), float(v)) for c, v in lst] for q, lst in d.items()} for s, d in _bm["lists"].items()}
del _bm
print("BM25 config:", BM25_CFG)
for s in FILES:
    miss = sum(q["key"] not in BM25[s] for q in QRY[s]); assert miss == 0, f"{s}: {miss} queries have no BM25 list"

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train: 827 queries | pool 4320 | queries inside their own pool: 827 | gold pairs 5308 | queries without gold 10


Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

... (3 outputs left out)

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

test : 237 queries | pool 1727 | queries inside their own pool: 237
distinct documents over all splits: 5825
BM25 config: {'ng': 'bi', 'k1': 8.0, 'b': 1.0, 'mode': 'binary', 'H@7': 395, 'F1@7': 0.43913285158421345, 'recall@100': 0.7543679342240494, 'MRR': 0.8950390337803802, 'year_slack': None}


## Tokenisation, views, cache
Every distinct document is tokenised once (pools overlap) into its **document view** (head 2/3 + tail 1/3 of `MAX_DOC_CHARS` characters, then at most `MAX_DOC_CHUNKS × 510` tokens), and every query additionally into its **evidence view** (the `<CITATION>` sentences ± `CTX_WINDOW`, in document order, at most `Q_MAX_CHUNKS × 510` tokens). The cache is keyed by the view settings and written to `WORK_DIR/tokcache/`. **Upload that folder as the Kaggle dataset `caps-p7-tokcache`**: every later Phase-7 run uses the same tokenizer and reads it instead of re-tokenising (~5–15 min saved per run).

In [15]:
# Cell 14 — Tokenise (multiprocessing) + cache; chunking
def cut_text(t):
    if len(t) <= MAX_DOC_CHARS: return t
    h = 2 * MAX_DOC_CHARS // 3
    return t[:h] + "\n" + t[-(MAX_DOC_CHARS - h):]
def ctx_text(sents):
    hit = [j for j, s in enumerate(sents) if CIT_TAG in s]
    keep = sorted({k for j in hit for k in range(max(0, j - CTX_WINDOW), min(len(sents), j + CTX_WINDOW + 1))})
    return "\n".join(sents[k] for k in keep)

_W_ITEMS = None
def _tok_work(i):
    k, want_ctx = _W_ITEMS[i]; sents = DOCS[k]["sents"]
    doc = np.asarray(tokenizer.encode(cut_text("\n".join(sents)))[:MAX_DOC_TOK], dtype=np.uint16)
    ctx = None
    if want_ctx:
        t = ctx_text(sents)
        ctx = np.asarray(tokenizer.encode(t[: int(Q_MAX_CHUNKS * BODY * 4.6)])[: Q_MAX_CHUNKS * BODY], dtype=np.uint16) if t else np.zeros(0, np.uint16)
    return k, doc, ctx

SIG = hashlib.md5(json.dumps([CHUNK_TOKENS, MAX_DOC_CHUNKS, CTX_WINDOW, Q_MAX_CHUNKS, CIT_TAG, VOCAB_SIZE]).encode()).hexdigest()[:10]
CACHE_NAME = f"p7_tok_{SIG}.pkl"
cache_out = WORK_DIR / "tokcache" / CACHE_NAME; cache_out.parent.mkdir(parents=True, exist_ok=True)
TOKD, TOKQ = {}, {}
for p in (TOKCACHE_IN / CACHE_NAME, cache_out):
    if p.exists():
        TOKD, TOKQ = pickle.loads(p.read_bytes()); print(f"token cache loaded: {p} ({len(TOKD)} docs, {len(TOKQ)} query views)"); break
qkeys = {q["key"] for s in FILES for q in QRY[s]}
todo = [(k, k in qkeys) for k in DOCS if k not in TOKD or (k in qkeys and k not in TOKQ)]
if todo:
    t0 = time.time(); _W_ITEMS = todo
    with mp.get_context("fork").Pool(N_PROC) as pool:
        for k, d, c in tqdm(pool.imap_unordered(_tok_work, range(len(todo)), chunksize=4), total=len(todo), desc="tokenise"):
            TOKD[k] = d
            if c is not None: TOKQ[k] = c
    cache_out.write_bytes(pickle.dumps((TOKD, TOKQ), protocol=4))
    print(f"tokenised {len(todo)} documents in {time.time() - t0:.0f}s -> {cache_out} ({cache_out.stat().st_size / 1e6:.0f} MB)")

def chunks_of(ids, max_chunks):
    ids = ids[: max_chunks * BODY]
    ch = [ids[i:i + BODY] for i in range(0, len(ids), BODY)]
    if len(ch) > 1 and len(ch[-1]) < 32: ch = ch[:-1]          # a 5-token tail chunk is noise
    return ch or [np.zeros(0, np.uint16)]
def doc_chunks(k): return chunks_of(TOKD[k], MAX_DOC_CHUNKS)
def query_chunks(k):
    head = chunks_of(TOKD[k], 1)
    ctx = chunks_of(TOKQ[k], Q_MAX_CHUNKS) if len(TOKQ.get(k, ())) else []
    if QUERY_VIEW == "head": return chunks_of(TOKD[k], Q_MAX_CHUNKS)
    if QUERY_VIEW == "ctx": return ctx or head
    return (head + ctx)[:Q_MAX_CHUNKS]
_nq = [len(query_chunks(q["key"])) for q in QRY["train"]]; _nd = [len(doc_chunks(k)) for k in POOL["train"]]
print(f"train: query chunks/query mean {np.mean(_nq):.2f} | doc chunks/doc mean {np.mean(_nd):.2f} (cap {MAX_DOC_CHUNKS}) | "
      f"queries with an empty ctx view: {sum(len(TOKQ.get(q['key'], ())) == 0 for q in QRY['train'])}")

token cache loaded: /kaggle/input/datasets/ascyrax/caps-p7-tokcache/p7_tok_18962ef646.pkl (5825 docs, 1182 query views)
train: query chunks/query mean 3.70 | doc chunks/doc mean 9.98 (cap 16) | queries with an empty ctx view: 1


## The adapted model, embedding, scoring, evaluation, fusion

In [16]:
# Cell 15 — Partial / layer-wise fine-tuning; hybrid causal / bidirectional attention; bi-encoder construction
def _bidir_alibi_bias(bb, B, T, mask, dev):
    # symmetric ALiBi -m_h |i-j| (no causal part) + key-padding mask (real keys always remain, so no all -inf rows)
    pos = torch.arange(T, device=dev)
    dist = (pos[:, None] - pos[None, :]).abs().float()
    bias = (-bb.alibi.slopes.to(dev)[:, None, None] * dist).unsqueeze(0)                       # [1, H, T, T]
    if mask is not None:
        bias = bias.expand(B, -1, -1, -1).masked_fill(~mask.bool()[:, None, None, :], float("-inf"))
    return bias

def build_model():
    # Fresh copy of the Phase-1 weights; top PARTIAL_K blocks + final norm + identity-initialised head trainable.
    model = copy.deepcopy(lm).cpu()
    for p in model.parameters(): p.requires_grad = False
    n = len(model.backbone.blocks); model.first_trainable = n - PARTIAL_K
    for blk in model.backbone.blocks[model.first_trainable:]:
        for p in blk.parameters(): p.requires_grad = True
    for p in model.backbone.norm_f.parameters(): p.requires_grad = True
    model.bidir_from = n                                           # start in the Phase-1 causal mode (zero-shot, identity check)
    d = lm_cfg.d_model
    model.head = nn.Linear(d, d, bias=False) if USE_HEAD else None
    if USE_HEAD:
        with torch.no_grad(): model.head.weight.copy_(torch.eye(d))
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"partial FT: blocks {model.first_trainable}..{n-1} of {n} trainable (+ norm_f, head) | params: total {tot/1e6:.2f}M | "
          f"trainable {tr/1e6:.2f}M ({100*tr/tot:.1f}%) | head {d*d/1e6 if USE_HEAD else 0:.3f}M")
    model.n_total, model.n_trainable = tot, tr
    # equivalence check (causal mode) against the Phase-1 Backbone.forward on unpadded input
    with torch.no_grad():
        _ids = torch.randint(5, VOCAB_SIZE, (2, 40), device=device)
        _a = backbone_hidden(model, _ids, None); _b, _ = lm.backbone(_ids)
    err = (_a - _b).abs().max().item()
    print(f"backbone_hidden (causal) == Phase-1 Backbone.forward: max diff {err:.2e}")
    assert err < 1e-3
    return model

def set_attention(model, mode):
    n = len(model.backbone.blocks)
    model.bidir_from = n - BIDIR_TOP_K if mode == "bidir_top" else n
    print("attention: all blocks causal (Phase-1)" if model.bidir_from >= n else
          f"attention: blocks 0..{model.bidir_from - 1} causal | blocks {model.bidir_from}..{n - 1} bidirectional (symmetric ALiBi)")

def backbone_hidden(model, ids, mask):
    # Right-padded chunks. Causal blocks: ALiBi + causal bias, no padding mask needed (real tokens never see the pads to their right).
    # Bidirectional blocks: symmetric ALiBi + key-padding mask. Frozen lower blocks run under no_grad (nothing stored for backward).
    bb = model.backbone; N, T = ids.shape; dev = ids.device
    nb = len(bb.blocks); bf = getattr(model, "bidir_from", nb); ft = getattr(model, "first_trainable", 0)
    b_c = Backbone._attn_bias(bb, N, T, 0, None, dev) if bf > 0 else None
    b_b = _bidir_alibi_bias(bb, N, T, mask, dev) if bf < nb else None
    train_grad = model.training and torch.is_grad_enabled()
    ft = ft if train_grad else 0                                   # blocks [0, ft) run without autograd during training
    with torch.no_grad() if ft > 0 else contextlib.nullcontext():
        x = bb.emb_drop(bb.tok_emb(ids))
        for i in range(ft):
            x, _ = bb.blocks[i](x, attn_bias=b_b if i >= bf else b_c, rope_cs=None, kv_cache=None)
    for i in range(ft, nb):
        blk, bias = bb.blocks[i], (b_b if i >= bf else b_c)
        if train_grad and GRAD_CKPT:
            x = checkpoint(lambda t, blk=blk, bias=bias: blk(t, attn_bias=bias, rope_cs=None, kv_cache=None)[0], x, use_reentrant=False)
        else:
            x, _ = blk(x, attn_bias=bias, rope_cs=None, kv_cache=None)
    return bb.norm_f(x)

In [17]:
# Cell 16 — Pooling, chunk embedding, document scoring (mean / maxsim), evaluation, fusion
def make_batch(seqs):
    # seqs: list of uint16 arrays (chunk bodies). Returns ids [N, T] right-padded with [BOS] body [EOS], lengths [N].
    lens = [len(s) + 2 for s in seqs]; T = max(lens)
    ids = torch.full((len(seqs), T), PAD_ID, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, 0] = BOS_ID; ids[i, 1:1 + len(s)] = torch.from_numpy(s.astype(np.int64)); ids[i, 1 + len(s)] = EOS_ID
    return ids.to(device, non_blocking=True), torch.tensor(lens, device=device)

def pool_hidden(h, lens, how):
    # h [N, T, d] (right padding), lens [N] -> [N, d]
    N, T, _ = h.shape
    t = torch.arange(T, device=h.device).unsqueeze(0)
    m = (t < lens.unsqueeze(1)).to(h.dtype)
    if how == "mean":  w = m
    elif how == "wmean": w = m * (t + 1).to(h.dtype)                 # SGPT position weights
    elif how == "last":  return h[torch.arange(N, device=h.device), lens - 1]
    else: raise ValueError(how)
    return (h * w.unsqueeze(-1)).sum(1) / w.sum(1, keepdim=True)

def encode(model, ids, lens, poolings):
    mask = (torch.arange(ids.shape[1], device=ids.device).unsqueeze(0) < lens.unsqueeze(1)).long()     # key-padding mask
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        h = backbone_hidden(model, ids, mask)
    out = {}
    for p in poolings:
        e = pool_hidden(h.float(), lens, p)
        if getattr(model, "head", None) is not None: e = model.head(e)
        out[p] = F.normalize(e, dim=-1)
    return out

@torch.no_grad()
def embed_keys(model, keys, chunk_fn, poolings):
    # -> {pooling: [n_chunks, d] float32 (GPU)}, owner [n_chunks] (index into keys), sorted by owner
    model.eval()
    seqs, owner = [], []
    for i, k in enumerate(keys):
        for c in chunk_fn(k): seqs.append(c); owner.append(i)
    order = np.argsort([len(s) for s in seqs], kind="stable")
    E = {p: torch.empty(len(seqs), lm_cfg.d_model, device=device) for p in poolings}
    for b in range(0, len(order), EMB_BATCH):
        idx = order[b:b + EMB_BATCH]
        ids, lens = make_batch([seqs[j] for j in idx])
        out = encode(model, ids, lens, poolings)
        for p in poolings: E[p][torch.as_tensor(idx, device=device)] = out[p]
    return E, torch.tensor(owner, device=device)

def embed_split(model, split, poolings, q_subset=None):
    qs = QRY[split] if q_subset is None else QRY[split][:q_subset]
    t0 = time.time()
    Eq, oq = embed_keys(model, [q["key"] for q in qs], query_chunks, poolings)
    Ed, od = embed_keys(model, POOL[split], doc_chunks, poolings)
    return dict(Eq=Eq, oq=oq, Ed=Ed, od=od, nq=len(qs), nd=len(POOL[split]), secs=time.time() - t0, n_chunks=len(od) + len(oq))

def seg_mean(E, owner, n):
    s = torch.zeros(n, E.shape[1], device=E.device).index_add_(0, owner, E)
    return F.normalize(s, dim=-1)

@torch.no_grad()
def dense_scores(emb, pooling, mode, center=False, block=256):
    Eq, Ed, oq, od = emb["Eq"][pooling], emb["Ed"][pooling], emb["oq"], emb["od"]
    if center:
        mu = Ed.mean(0, keepdim=True); Eq = F.normalize(Eq - mu, dim=-1); Ed = F.normalize(Ed - mu, dim=-1)
    nq, nd = emb["nq"], emb["nd"]
    if mode == "mean":
        S = seg_mean(Eq, oq, nq) @ seg_mean(Ed, od, nd).T
    else:                                               # maxsim: mean over query chunks of the max over candidate chunks
        S = torch.zeros(nq, nd, device=device); cnt = torch.zeros(nq, device=device)
        for b in range(0, len(oq), block):
            sim = Eq[b:b + block] @ Ed.T                                        # [qb, n_doc_chunks]
            mx = torch.full((sim.shape[0], nd), -2.0, device=device).scatter_reduce(1, od.expand(sim.shape[0], -1), sim, "amax")
            S.index_add_(0, oq[b:b + block], mx); cnt.index_add_(0, oq[b:b + block], torch.ones(len(mx), device=device))
        S = S / cnt.clamp(min=1).unsqueeze(1)
    return S.float().cpu().numpy()

def self_mask(S, split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; S = S.copy()
    for i, q in enumerate(QRY[split][: S.shape[0]]):
        if q["key"] in col and not (q["gold"] and q["key"] in q["gold"]): S[i, col[q["key"]]] = -np.inf
    return S

def bm25_matrix(split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; qs = QRY[split][:nq] if nq else QRY[split]
    S = np.zeros((len(qs), len(POOL[split])), dtype=np.float32)
    for i, q in enumerate(qs):
        lst = BM25[split][q["key"]]; lo = min(v for _, v in lst) if lst else 0.0
        S[i, :] = lo - 1.0
        for c, v in lst:
            if c in col: S[i, col[c]] = v
    return S

def topk(S, k):
    k = min(k, S.shape[1]); idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    o = np.argsort(-np.take_along_axis(S, idx, 1), axis=1); return np.take_along_axis(idx, o, 1)
def gold_cols(split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; qs = QRY[split][:nq] if nq else QRY[split]
    return [set(col[g] for g in q["gold"]) for q in qs]
def evaluate(S, split, ks=(K_EVAL,), recall_ks=RECALL_KS):
    gc_ = gold_cols(split, S.shape[0]); N = len(gc_); G = sum(len(g) for g in gc_)
    T = topk(S, max(max(ks), max(recall_ks))); out = {}
    for k in ks:
        H = sum(len(set(T[i, :k].tolist()) & gc_[i]) for i in range(N))
        out[f"H@{k}"] = H; out[f"P@{k}"] = H / (k * N); out[f"R@{k}"] = H / max(G, 1); out[f"F1@{k}"] = 2 * H / (k * N + G)
    for k in recall_ks: out[f"recall@{k}"] = sum(len(set(T[i, :k].tolist()) & gc_[i]) for i in range(N)) / max(G, 1)
    rr = [next((1 / (j + 1) for j, c in enumerate(T[i].tolist()) if c in gc_[i]), 0.0) for i in range(N)]
    out["MRR"] = float(np.mean(rr)); return out

def znorm(S):
    S = S.astype(np.float64); f = np.isfinite(S)
    m = np.where(f, S, 0).sum(1, keepdims=True) / f.sum(1, keepdims=True)
    v = np.where(f, (S - m) ** 2, 0).sum(1, keepdims=True) / f.sum(1, keepdims=True)
    return np.where(f, (S - m) / np.sqrt(v + 1e-12), -np.inf)
def rrf(*Ss):
    out = np.zeros_like(Ss[0], dtype=np.float64)
    for S in Ss:
        r = np.empty_like(S, dtype=np.int64); o = np.argsort(-S, axis=1); r[np.arange(len(S))[:, None], o] = np.arange(S.shape[1])
        out += 1.0 / (RRF_K + 1 + r)
    return out
def fuse_search(Sd, Sb, split):
    # Sd, Sb already self-masked. Returns best (alpha, eval) by H@7 (ties: recall@100), and the whole curve.
    zd, zb = znorm(Sd), znorm(Sb); curve = []
    for a in ALPHA_GRID:
        e = evaluate(a * np.nan_to_num(zd, neginf=-1e9) + (1 - a) * np.nan_to_num(zb, neginf=-1e9), split)
        curve.append((float(a), e["H@7"], e["F1@7"], e["recall@100"]))
    best = max(curve, key=lambda t: (t[1], t[3], -abs(t[0] - 0.5)))
    return best, curve
def fused(Sd, Sb, alpha):
    zd, zb = znorm(Sd), znorm(Sb)
    return alpha * np.nan_to_num(zd, neginf=-1e9) + (1 - alpha) * np.nan_to_num(zb, neginf=-1e9)

## Zero-shot transfer of the Phase-1 representations (before any training)
One pass over dev produces all three poolings (they share the hidden states). For each pooling and score mode we report **dense-only** dev metrics, the **best fusion with BM25**, and the effect of **centering** (subtracting the pool's mean chunk vector before the cosine). Centering targets *anisotropy*: language-model hidden states share a large common direction, so all cosines are close to each other and dominated by it (Ethayarajh 2019); removing the mean exposes the directions that actually differ between documents. If centering helps a lot zero-shot, that is direct evidence of anisotropy in the Phase-1 model.

In [18]:
# Cell 17 — Build the model; BM25-only dev reference; zero-shot table; choose POOLING
seed_everything(SEED)
model = build_model()
DEV_Q = SMOKE_DEV_Q if SMOKE_TEST else None
Sb_dev = self_mask(bm25_matrix("dev", DEV_Q), "dev")
e_b = evaluate(Sb_dev, "dev")
print(f"BM25 only (dev{'[:%d]' % DEV_Q if DEV_Q else ''}): H@7 {e_b['H@7']} F1@7 {e_b['F1@7']:.4f} recall@100 {e_b['recall@100']:.3f} MRR {e_b['MRR']:.3f}")

POOLS_ALL = ("mean", "wmean", "last")
emb0 = embed_split(model, "dev", POOLS_ALL, DEV_Q)
print(f"dev embedded: {emb0['n_chunks']} chunks in {emb0['secs']:.0f}s")
ZS = []
for p in POOLS_ALL:
    for mode in ("mean", "maxsim"):
        for center in (False, True):
            Sd = self_mask(dense_scores(emb0, p, mode, center), "dev")
            e = evaluate(Sd, "dev"); (a, H, f1, r100), _ = fuse_search(Sd, Sb_dev, "dev")
            ZS.append(dict(pooling=p, mode=mode, center=center, H7=e["H@7"], F1=e["F1@7"], r100=e["recall@100"], MRR=e["MRR"],
                           fused_H7=H, fused_F1=f1, alpha=a))
            print(f"  zero-shot {p:5s} {mode:6s} center={center!s:5s}: dense H@7 {e['H@7']:4d} F1 {e['F1@7']:.4f} r@100 {e['recall@100']:.3f} | "
                  f"fused H@7 {H:4d} F1 {f1:.4f} (alpha {a})")
if POOLING == "auto":
    POOLING = max(POOLS_ALL, key=lambda p: max(r["H7"] for r in ZS if r["pooling"] == p))
print("POOLING ->", POOLING)
del emb0; gc.collect(); torch.cuda.empty_cache() if device.type == "cuda" else None

partial FT: blocks 9..12 of 13 trainable (+ norm_f, head) | params: total 54.15M | trainable 11.80M (21.8%) | head 0.262M
backbone_hidden (causal) == Phase-1 Backbone.forward: max diff 0.00e+00
BM25 only (dev): H@7 395 F1@7 0.4391 recall@100 0.754 MRR 0.895
dev embedded: 11164 chunks in 50s
  zero-shot mean  mean   center=False: dense H@7  217 F1 0.2412 r@100 0.633 | fused H@7  396 F1 0.4402 (alpha 0.1)
  zero-shot mean  mean   center=True : dense H@7  228 F1 0.2535 r@100 0.645 | fused H@7  396 F1 0.4402 (alpha 0.1)
  zero-shot mean  maxsim center=False: dense H@7  216 F1 0.2401 r@100 0.625 | fused H@7  398 F1 0.4425 (alpha 0.3)
  zero-shot mean  maxsim center=True : dense H@7  231 F1 0.2568 r@100 0.630 | fused H@7  397 F1 0.4414 (alpha 0.3)
  zero-shot wmean mean   center=False: dense H@7  237 F1 0.2635 r@100 0.649 | fused H@7  396 F1 0.4402 (alpha 0.2)
  zero-shot wmean mean   center=True : dense H@7  233 F1 0.2590 r@100 0.646 | fused H@7  396 F1 0.4402 (alpha 0.1)
  zero-shot wmean 

## Training
Each step: `BATCH_PAIRS` (query, positive) pairs from a shuffled list of all train gold pairs (an epoch visits every gold pair once, so queries with many citations get proportionally more steps — consistent with the micro-averaged metric). For each pair one hard negative is drawn from the query's BM25 top-`HN_TOP` minus its gold and itself. Every document contributes `C_TRAIN` random chunks and every query `QC_TRAIN` random evidence-view chunks (fresh random choices every step = augmentation). Embeddings are the normalised means of the chunk vectors. The forward runs in pieces of `FWD_MICRO` sequences with activation checkpointing; the loss needs all `2B` documents together, so the backward is one call.

In [19]:
# Cell 18 — Training loop (multi-positive InfoNCE with false-negative masking), per-epoch dev evaluation
TRQ = [q for q in QRY["train"] if q["gold"]]
TR_PAIRS = [(qi, g) for qi, q in enumerate(TRQ) for g in q["gold"]]
TR_GOLD = [set(q["gold"]) for q in TRQ]
TR_HN = [[c for c, _ in BM25["train"][q["key"]][:HN_TOP] if c not in TR_GOLD[qi] and c != q["key"]] for qi, q in enumerate(TRQ)]
print(f"train: {len(TRQ)} queries with gold | {len(TR_PAIRS)} (query, positive) pairs | queries without a hard negative: {sum(not h for h in TR_HN)}")
_rank = [{c: r + 1 for r, (c, _) in enumerate(BM25["train"][q["key"]])} for q in TRQ]
TR_POS_RANK = [_rank[qi].get(g, 10**6) for qi, g in TR_PAIRS]
TR_W = [1.0 + (HARD_BOOST if r > HARD_RANK else 0.0) for r in TR_POS_RANK]
print(f"train positives ranked by BM25: top-7 {np.mean([r <= 7 for r in TR_POS_RANK]):.1%} | beyond {HARD_RANK}: "
      f"{np.mean([r > HARD_RANK for r in TR_POS_RANK]):.1%} | beyond 300: {np.mean([r > 300 for r in TR_POS_RANK]):.1%} | POS_SAMPLING={POS_SAMPLING}")

# switch the top blocks to bidirectional attention (if configured) and measure step 0 in that mode
STEP0 = {}
if ATTN_MODE == "bidir_top":
    set_attention(model, "bidir_top")
    if POOLING != "mean":
        print(f"bidir_top: every final-layer position sees the whole chunk -> POOLING '{POOLING}' replaced by 'mean'"); POOLING = "mean"
    STEP0_PENDING = True
else:
    STEP0_PENDING = False

def sample_chunks(ch, n, rng): return ch if len(ch) <= n else [ch[i] for i in sorted(rng.sample(range(len(ch)), n))]

def step_loss(model, batch, rng):
    # batch: list of (qi, pos_key). Returns loss, stats.
    docs = []
    for qi, pk in batch:
        docs.append(pk)
        hn = TR_HN[qi]; docs.append(rng.choice(hn) if hn else rng.choice(POOL["train"]))
    seqs, owner = [], []
    for i, (qi, _) in enumerate(batch):
        for c in sample_chunks(query_chunks(TRQ[qi]["key"]), QC_TRAIN, rng): seqs.append(c); owner.append(i)
    nQ = len(batch)
    for j, k in enumerate(docs):
        for c in sample_chunks(doc_chunks(k), C_TRAIN, rng): seqs.append(c); owner.append(nQ + j)
    embs = []
    for b in range(0, len(seqs), FWD_MICRO):
        ids, lens = make_batch(seqs[b:b + FWD_MICRO]); embs.append(encode(model, ids, lens, (POOLING,))[POOLING])
    E = torch.cat(embs); own = torch.tensor(owner, device=device)
    V = seg_mean(E, own, nQ + len(docs)); Q, D = V[:nQ], V[nQ:]
    logits = (Q @ D.T) / TAU                                                  # [B, 2B]
    mask = torch.zeros_like(logits, dtype=torch.bool)
    for i, (qi, _) in enumerate(batch):
        for j, k in enumerate(docs):
            if j != 2 * i and (k in TR_GOLD[qi] or k == TRQ[qi]["key"]): mask[i, j] = True
    logits = logits.masked_fill(mask, float("-inf"))
    target = torch.arange(nQ, device=device) * 2
    loss = F.cross_entropy(logits.float(), target)
    with torch.no_grad():
        acc = (logits.argmax(1) == target).float().mean().item()
        gap = (logits[torch.arange(nQ), target] - logits[torch.arange(nQ), target + 1]).mean().item() * TAU   # cos(q,pos) - cos(q,hard neg)
    return loss, dict(acc=acc, gap=gap, masked=int(mask.sum().item()), seqs=len(seqs))

def param_groups(model):
    # Layer-wise rates: block i gets lr * decay^(n-1-i); decoupled weight decay on 2-D matrices only; head at head_lr.
    hp = HP[METHOD]; nb = len(model.backbone.blocks); g = []
    for i in range(model.first_trainable, nb):
        lr_i = hp["lr"] * hp["decay"] ** (nb - 1 - i)
        ps = [p for p in model.backbone.blocks[i].parameters() if p.requires_grad]
        g.append(dict(params=[p for p in ps if p.ndim >= 2], lr=lr_i, base_lr=lr_i, weight_decay=hp["wd"]))
        g.append(dict(params=[p for p in ps if p.ndim < 2], lr=lr_i, base_lr=lr_i, weight_decay=0.0))
    g.append(dict(params=list(model.backbone.norm_f.parameters()), lr=hp["lr"], base_lr=hp["lr"], weight_decay=0.0))
    if getattr(model, "head", None) is not None:
        g.append(dict(params=list(model.head.parameters()), lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0))
    print("layer-wise lr:", {i: f"{hp['lr'] * hp['decay'] ** (nb - 1 - i):.2e}" for i in range(model.first_trainable, nb)},
          "| head", hp["head_lr"])
    return [x for x in g if x["params"]]

def dev_eval(model, emb=None):
    emb = emb or embed_split(model, "dev", (POOLING,), DEV_Q)
    out = {}
    for mode in ("mean", "maxsim"):
        Sd = self_mask(dense_scores(emb, POOLING, mode), "dev")
        e = evaluate(Sd, "dev"); (a, H, f1, r100), _ = fuse_search(Sd, Sb_dev, "dev")
        out[mode] = dict(H7=e["H@7"], F1=e["F1@7"], r100=e["recall@100"], fused_H7=H, fused_F1=f1, alpha=a)
    return out, emb

def train_model(model, epochs, max_steps=None):
    opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]
    spe = math.ceil(len(TR_PAIRS) / BATCH_PAIRS); total = epochs * spe
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))
    print(f"{spe} steps/epoch | total {total} steps | warmup {warm}")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, step, done, t_train = [], 0, False, time.time()
    best = dict(score=None, epoch=0, state=None)
    for ep in range(1, epochs + 1):
        rng = random.Random(SEED + ep)
        if POS_SAMPLING == "bm25_hard": pairs = rng.choices(TR_PAIRS, weights=TR_W, k=len(TR_PAIRS))
        else: pairs = TR_PAIRS[:]; rng.shuffle(pairs)
        model.train(); t0 = time.time(); L, A, Gp = [], [], []
        for b in range(0, len(pairs), BATCH_PAIRS):
            loss, st = step_loss(model, pairs[b:b + BATCH_PAIRS], rng)
            m = lr_mult(step)
            for g in opt.param_groups: g["lr"] = g["base_lr"] * m
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
            L.append(loss.item()); A.append(st["acc"]); Gp.append(st["gap"])
            if step == 10:
                per = (time.time() - t0) / 10
                print(f"  >> speed {per:.2f} s/step ({st['seqs']} seqs) | projected {per * spe / 60:.1f} min/epoch"
                      + (f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device.type == "cuda" else ""))
            if step % 50 == 0:
                print(f"  ep {ep} step {step}/{total} | loss {np.mean(L[-50:]):.4f} | in-batch acc {np.mean(A[-50:]):.3f} | "
                      f"cos(pos)-cos(hn) {np.mean(Gp[-50:]):+.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
            if max_steps and step >= max_steps: done = True; break
            if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping"); done = True; break
        ev, _ = dev_eval(model)
        score = max(ev[m]["fused_H7"] for m in ev) + 1e-3 * max(ev[m]["H7"] for m in ev)
        rec = dict(epoch=ep, loss=float(np.mean(L)), acc=float(np.mean(A)), gap=float(np.mean(Gp)), steps=step, secs=time.time() - t0, dev=ev)
        history.append(rec)
        print(f"epoch {ep}: loss {rec['loss']:.4f} acc {rec['acc']:.3f} | dev dense H@7 mean {ev['mean']['H7']} / maxsim {ev['maxsim']['H7']} | "
              f"fused H@7 mean {ev['mean']['fused_H7']} (a={ev['mean']['alpha']}) / maxsim {ev['maxsim']['fused_H7']} (a={ev['maxsim']['alpha']})")
        if best["score"] is None or score > best["score"]:
            best = dict(score=score, epoch=ep, state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
            torch.save(best, RUN_DIR / "best_trainable.pt")
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0)
    return history, best, stats

if STEP0_PENDING:
    STEP0, _ = dev_eval(model)
    print(f"step 0 in bidir_top (no training yet): dense H@7 mean {STEP0['mean']['H7']} / maxsim {STEP0['maxsim']['H7']} | "
          f"fused {STEP0['mean']['fused_H7']} / {STEP0['maxsim']['fused_H7']}   (compare with the causal zero-shot table)")
history, best, stats = train_model(model, EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)
print(f"\nbest epoch {best['epoch']} | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

train: 817 queries with gold | 5308 (query, positive) pairs | queries without a hard negative: 0
train positives ranked by BM25: top-7 36.6% | beyond 50: 30.7% | beyond 300: 15.2% | POS_SAMPLING=uniform
attention: blocks 0..8 causal | blocks 9..12 bidirectional (symmetric ALiBi)
bidir_top: every final-layer position sees the whole chunk -> POOLING 'wmean' replaced by 'mean'
step 0 in bidir_top (no training yet): dense H@7 mean 219 / maxsim 224 | fused 396 / 397   (compare with the causal zero-shot table)
layer-wise lr: {9: '5.12e-05', 10: '6.40e-05', 11: '8.00e-05', 12: '1.00e-04'} | head 0.00025
166 steps/epoch | total 664 steps | warmup 39
  >> speed 3.16 s/step (303 seqs) | projected 8.7 min/epoch | peak GPU 5.4 GB
  ep 1 step 50/664 | loss 2.6657 | in-batch acc 0.313 | cos(pos)-cos(hn) -0.0123 | lr x0.999 | 157s
  ep 1 step 100/664 | loss 2.3491 | in-batch acc 0.382 | cos(pos)-cos(hn) -0.0007 | lr x0.979 | 314s
  ep 1 step 150/664 | loss 2.3850 | in-batch acc 0.374 | cos(pos)-cos(h

## Final dev evaluation and analysis
With the best epoch restored: dense-only and fused scores for both score modes (the better one becomes `SCORE_MODE`), the full `α` curve, reciprocal-rank fusion, the F1@K curve, the rank of every gold case under the fused score (versus under BM25 alone), how many gold cases the fused ranking **adds** to BM25's top 7 and how many it **loses**, and the per-query hit distribution.

In [20]:
# Cell 19 — Final dev evaluation: score mode, alpha curve, RRF, F1@K, gold ranks, gains vs BM25
ev_final, emb_dev = dev_eval(model)
if SCORE_MODE == "auto":
    SCORE_MODE = max(("mean", "maxsim"), key=lambda m: (ev_final[m]["fused_H7"], ev_final[m]["H7"]))
Sd_dev = self_mask(dense_scores(emb_dev, POOLING, SCORE_MODE), "dev")
(ALPHA, H_f, F1_f, r100_f), curve = fuse_search(Sd_dev, Sb_dev, "dev")
Sf_dev = fused(Sd_dev, Sb_dev, ALPHA)
e_d, e_f = evaluate(Sd_dev, "dev"), evaluate(Sf_dev, "dev", ks=tuple(range(1, 21)))
e_rrf = evaluate(rrf(Sd_dev, Sb_dev), "dev")
print(f"SCORE_MODE={SCORE_MODE} | POOLING={POOLING} | ALPHA={ALPHA}")
print(f"  BM25 only : H@7 {e_b['H@7']:4d} F1@7 {e_b['F1@7']:.4f} recall@100 {e_b['recall@100']:.3f} MRR {e_b['MRR']:.3f}")
print(f"  dense only: H@7 {e_d['H@7']:4d} F1@7 {e_d['F1@7']:.4f} recall@100 {e_d['recall@100']:.3f} MRR {e_d['MRR']:.3f}")
print(f"  fused     : H@7 {e_f['H@7']:4d} F1@7 {e_f['F1@7']:.4f} recall@100 {e_f['recall@100']:.3f} MRR {e_f['MRR']:.3f}  (P@7 {e_f['P@7']:.3f} R@7 {e_f['R@7']:.3f})")
print(f"  RRF       : H@7 {e_rrf['H@7']:4d} F1@7 {e_rrf['F1@7']:.4f} recall@100 {e_rrf['recall@100']:.3f}")
print("  alpha curve (alpha, H@7):", [(a, H) for a, H, _, _ in curve])
print("  F1@K (fused):", {k: round(e_f[f'F1@{k}'], 4) for k in range(1, 21)})

gcols = gold_cols("dev", Sd_dev.shape[0])
def ranks(S):
    o = np.argsort(-S, axis=1); r = np.empty_like(o); r[np.arange(len(o))[:, None], o] = np.arange(o.shape[1])
    return np.asarray([r[i, c] + 1 for i, g in enumerate(gcols) for c in g])
rb, rf, rd = ranks(Sb_dev), ranks(Sf_dev), ranks(Sd_dev)
print(f"  gold rank median: BM25 {np.median(rb):.0f} | dense {np.median(rd):.0f} | fused {np.median(rf):.0f}  "
      f"(top-100 share: {np.mean(rb <= 100):.3f} / {np.mean(rd <= 100):.3f} / {np.mean(rf <= 100):.3f})")
Tb, Tf = topk(Sb_dev, 7), topk(Sf_dev, 7)
gain = sum(len((set(Tf[i].tolist()) - set(Tb[i].tolist())) & gcols[i]) for i in range(len(gcols)))
loss_ = sum(len((set(Tb[i].tolist()) - set(Tf[i].tolist())) & gcols[i]) for i in range(len(gcols)))
hits = np.asarray([len(set(Tf[i].tolist()) & gcols[i]) for i in range(len(gcols))])
print(f"  vs BM25 top-7: gold cases gained {gain} | lost {loss_} | net {gain - loss_:+d}")
print(f"  hits@7 per query (fused): {dict(sorted(Counter(hits.tolist()).items()))} | zero-hit queries {(hits == 0).mean():.1%}")
FINAL = dict(score_mode=SCORE_MODE, pooling=POOLING, alpha=ALPHA, bm25=e_b, dense=e_d,
             fused={k: v for k, v in e_f.items() if k.endswith("@7") or k.startswith("recall") or k == "MRR"},
             f1_curve={k: e_f[f"F1@{k}"] for k in range(1, 21)}, rrf=e_rrf, alpha_curve=curve,
             gold_rank_median=dict(bm25=float(np.median(rb)), dense=float(np.median(rd)), fused=float(np.median(rf))),
             gained=gain, lost=loss_, zero_hit=float((hits == 0).mean()), by_epoch_modes=ev_final)

SCORE_MODE=maxsim | POOLING=mean | ALPHA=0.55
  BM25 only : H@7  395 F1@7 0.4391 recall@100 0.754 MRR 0.895
  dense only: H@7  301 F1@7 0.3346 recall@100 0.728 MRR 0.727
  fused     : H@7  403 F1@7 0.4480 recall@100 0.787 MRR 0.901  (P@7 0.488 R@7 0.414)
  RRF       : H@7  365 F1@7 0.4058 recall@100 0.787
  alpha curve (alpha, H@7): [(0.0, 395), (0.05, 396), (0.1, 397), (0.15, 399), (0.2, 399), (0.25, 399), (0.3, 399), (0.35, 399), (0.4, 399), (0.45, 401), (0.5, 400), (0.55, 403), (0.6, 402), (0.65, 398), (0.7, 398), (0.75, 391), (0.8, 388), (0.85, 383), (0.9, 368), (0.95, 350), (1.0, 301)]
  F1@K (fused): {1: 0.1852, 2: 0.2911, 3: 0.3617, 4: 0.3958, 5: 0.4299, 6: 0.445, 7: 0.448, 8: 0.4455, 9: 0.4364, 10: 0.4329, 11: 0.4306, 12: 0.4203, 13: 0.4116, 14: 0.3985, 15: 0.3843, 16: 0.3775, 17: 0.3672, 18: 0.3591, 19: 0.3527, 20: 0.3438}
  gold rank median: BM25 12 | dense 22 | fused 11  (top-100 share: 0.754 / 0.728 / 0.787)
  vs BM25 top-7: gold cases gained 19 | lost 11 | net +8
  hits@7 

In [21]:
# Cell 20 — Test: embed, fuse with BM25, prediction.json -> zip (full runs only)
def validate_prediction(pred, split="test"):
    pool_ids = {DOCS[k]["id"] for k in POOL[split]}; qids = [q["id"] for q in QRY[split]]
    assert set(pred) == set(qids), f"missing/extra queries: {len(set(qids) - set(pred))} missing, {len(set(pred) - set(qids))} extra"
    for qid, lst in pred.items():
        assert isinstance(lst, list) and len(lst) == K_EVAL and len(set(lst)) == K_EVAL, f"{qid}: need {K_EVAL} unique ids"
        assert all(isinstance(c, str) and c in pool_ids for c in lst), f"{qid}: id not in the {split} pool"
    return True

TEST = {}
if not SMOKE_TEST:
    t0 = time.time()
    emb_test = embed_split(model, "test", (POOLING,))
    Sd_test = self_mask(dense_scores(emb_test, POOLING, SCORE_MODE), "test")
    Sb_test = self_mask(bm25_matrix("test"), "test")
    Sf_test = fused(Sd_test, Sb_test, ALPHA)
    T7 = topk(Sf_test, K_EVAL)
    PRED = {q["id"]: [DOCS[POOL["test"][j]]["id"] for j in T7[i].tolist()] for i, q in enumerate(QRY["test"])}
    validate_prediction(PRED)
    sub_dir = RUN_DIR / "submission"; sub_dir.mkdir(exist_ok=True)
    (sub_dir / "prediction.json").write_text(json.dumps(PRED, indent=1))
    zpath = RUN_DIR / f"submission_{RUN_NAME}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z: z.write(sub_dir / "prediction.json", arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z: assert z.namelist() == ["prediction.json"]
    Tb7 = topk(Sb_test, K_EVAL)
    overlap = np.mean([len(set(T7[i].tolist()) & set(Tb7[i].tolist())) / K_EVAL for i in range(len(T7))])
    TEST = dict(seconds=time.time() - t0, embed_seconds=emb_test["secs"], chunks=emb_test["n_chunks"], top7_overlap_with_bm25=float(overlap))
    print(f"test: {len(PRED)} queries -> {zpath} | embed {emb_test['secs']:.0f}s ({emb_test['n_chunks']} chunks) | "
          f"top-7 overlap with BM25 {overlap:.2f}")
    if SAVE_SCORES:
        def top_list(S, split):
            T = topk(S, SAVE_TOP)
            return {q["id"]: [[DOCS[POOL[split][j]]["id"], round(float(S[i, j]), 5)] for j in T[i].tolist() if np.isfinite(S[i, j])]
                    for i, q in enumerate(QRY[split][: S.shape[0]])}
        (RUN_DIR / "dense_scores.json").write_text(json.dumps(dict(run=RUN_NAME, pooling=POOLING, mode=SCORE_MODE, alpha=ALPHA,
                                                                   dev=top_list(Sd_dev, "dev"), test=top_list(Sd_test, "test"))))
        print("saved dense top-%d scores for dev/test" % SAVE_TOP)

test: 237 queries -> /kaggle/working/runs/p7_dec03_partial/submission_p7_dec03_partial.zip | embed 91s (18838 chunks) | top-7 overlap with BM25 0.92
saved dense top-300 scores for dev/test


## KD teacher export (for the distillation run)
For every train query: the teacher's **raw cosine scores** (chosen pooling and score mode) over its BM25 top-300 ∪ its gold set, keyed by the pool's ID strings, together with `τ`. The student run turns them into a listwise target `softmax(s/τ)` over the same candidate list (and pairwise margins), so it learns the teacher's *ranking*, not just the binary gold labels. Only cosine similarities are exported, so the student can be any architecture with any embedding size.

In [22]:
# Cell 21 — Save trainable weights + log; teacher export
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    if isinstance(o, Path): return str(o)
    return o
torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, RUN_DIR / "trainable_final.pt")
log = dict(run=RUN_NAME, method=METHOD, hp=HP[METHOD], attn_mode=ATTN_MODE, step0=STEP0, pos_sampling=POS_SAMPLING, view=dict(chunk=CHUNK_TOKENS, max_doc_chunks=MAX_DOC_CHUNKS, query_view=QUERY_VIEW,
           ctx_window=CTX_WINDOW, q_max_chunks=Q_MAX_CHUNKS), train=dict(epochs=EPOCHS, B=BATCH_PAIRS, C=C_TRAIN, QC=QC_TRAIN, hn_top=HN_TOP, tau=TAU),
           params_total=model.n_total, params_trainable=model.n_trainable, zero_shot=ZS, history=history, best_epoch=best["epoch"],
           stats=stats, final=FINAL, test=TEST, bm25_cfg=BM25_CFG)
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=1))

TEACHER = {}
if EXPORT_TEACHER and not SMOKE_TEST:
    t0 = time.time()
    emb_tr = embed_split(model, "train", (POOLING,))
    S_tr = self_mask(dense_scores(emb_tr, POOLING, SCORE_MODE), "train")
    col = {k: j for j, k in enumerate(POOL["train"])}
    exp = {}
    for i, q in enumerate(QRY["train"]):
        cands = [c for c, _ in BM25["train"][q["key"]]] + [g for g in (q["gold"] or [])]
        seen, row = set(), []
        for c in cands:
            if c in seen or c == q["key"] or c not in col: continue
            seen.add(c); row.append([DOCS[c]["id"], round(float(S_tr[i, col[c]]), 5)])
        exp[q["id"]] = row
    tpath = RUN_DIR / f"teacher_{RUN_NAME}.json"
    tpath.write_text(json.dumps(dict(run=RUN_NAME, tau=TAU, pooling=POOLING, mode=SCORE_MODE, alpha=ALPHA, scores=exp)))
    TEACHER = dict(seconds=time.time() - t0, queries=len(exp), mean_list=float(np.mean([len(v) for v in exp.values()])),
                   mb=tpath.stat().st_size / 1e6)
    print(f"teacher export: {TEACHER} -> {tpath}")
for f in sorted(RUN_DIR.iterdir()): print(f"   {f.name:40s} {f.stat().st_size / 1e6 if f.is_file() else 0:8.2f} MB")

teacher export: {'seconds': 223.18353581428528, 'queries': 827, 'mean_list': 300.9782345828295, 'mb': 6.20827} -> /kaggle/working/runs/p7_dec03_partial/teacher_p7_dec03_partial.json
   best_trainable.pt                           47.22 MB
   dense_scores.json                            2.66 MB
   run_log.json                                 0.01 MB
   submission                                   0.00 MB
   submission_p7_dec03_partial.zip              0.01 MB
   teacher_p7_dec03_partial.json                6.21 MB
   trainable_final.pt                          47.22 MB


In [23]:
# Cell 22 — PASTE TO CLAUDE
def _r(o, n=4):
    if isinstance(o, dict): return {k: _r(v, n) for k, v in o.items()}
    if isinstance(o, list): return [_r(v, n) for v in o]
    if isinstance(o, float): return float(f"{o:.3g}") if 0 < abs(o) < 1e-2 else round(o, n)
    return o
short = dict(run=RUN_NAME, smoke=SMOKE_TEST, hp=HP[METHOD], attn_mode=ATTN_MODE, step0=STEP0, pos_sampling=POS_SAMPLING, params_trainable=model.n_trainable, params_total=model.n_total,
             zero_shot=ZS, history=[{k: v for k, v in h.items()} for h in history], best_epoch=best["epoch"], stats=stats,
             final={k: v for k, v in FINAL.items() if k not in ("alpha_curve",)}, alpha_curve=[(a, H) for a, H, _, _ in FINAL["alpha_curve"]],
             test=TEST, teacher=TEACHER)
print("================ PASTE TO CLAUDE ================")
print(json.dumps(_r(_jsonable(short)), separators=(",", ":")))
print("=================================================")

================ PASTE TO CLAUDE ================
{"run":"p7_dec03_partial","smoke":false,"hp":{"lr":0.0001,"head_lr":0.00025,"wd":0.01,"top_k":4,"decay":0.8},"attn_mode":"bidir_top","step0":{"mean":{"H7":219,"F1":0.2435,"r100":0.6362,"fused_H7":396,"fused_F1":0.4402,"alpha":0.35},"maxsim":{"H7":224,"F1":0.249,"r100":0.631,"fused_H7":397,"fused_F1":0.4414,"alpha":0.3}},"pos_sampling":"uniform","params_trainable":11801600,"params_total":54148224,"zero_shot":[{"pooling":"mean","mode":"mean","center":false,"H7":217,"F1":0.2412,"r100":0.6331,"MRR":0.5758,"fused_H7":396,"fused_F1":0.4402,"alpha":0.1},{"pooling":"mean","mode":"mean","center":true,"H7":228,"F1":0.2535,"r100":0.6454,"MRR":0.598,"fused_H7":396,"fused_F1":0.4402,"alpha":0.1},{"pooling":"mean","mode":"maxsim","center":false,"H7":216,"F1":0.2401,"r100":0.6249,"MRR":0.6239,"fused_H7":398,"fused_F1":0.4425,"alpha":0.3},{"pooling":"mean","mode":"maxsim","center":true,"H7":231,"F1":0.2568,"r100":0.63,"MRR":0.6424,"fused_H7":397,"fused